# 09 — Conventional laser-welding grouped ML evaluation

This notebook evaluates the conventional Al–Cu laser-welding dataset separately from wobble welding. It uses the 21 records that pass the current target-specific eligibility rule (the six all-zero geometry records remain preserved in the master dataset but are excluded from default geometry regression). The three repeats at each laser-power/speed condition are kept together during validation.

**Input:** `conventional_laser_welding_standardized.csv` from Notebook 03, or the conventional target-specific CSV exported by Notebook 06. This notebook accepts either and applies the same conservative all-zero exclusion.

**Validation:** leave-one-process-condition-out cross-validation (7 conditions; 3 repeats per condition). Results are exploratory because there are only seven independent process settings.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Upload the standardized conventional CSV if it is not already available.
candidate_files = [p for p in Path('.').glob('*.csv') if 'conventional_laser_welding_standardized' in p.name.lower()]
if not candidate_files:
    try:
        from google.colab import files
        uploaded = files.upload()
        candidate_files = [Path(name) for name in uploaded if 'conventional_laser_welding_standardized' in name.lower() and name.lower().endswith('.csv')]
    except ImportError:
        raise FileNotFoundError('Place conventional_laser_welding_standardized.csv in the working directory.')
if not candidate_files:
    raise FileNotFoundError('Upload conventional_laser_welding_standardized.csv (filename suffixes such as (2) are accepted).')
input_file = candidate_files[0]
df_raw = pd.read_csv(input_file)
print('Loaded:', input_file, '| rows:', len(df_raw))
display(df_raw.head())

## 1. Validate columns and define condition groups

Only laser power and welding speed are used as inputs because these are the documented process variables in this source. The condition group is the exact power–speed combination.

In [ ]:
features = ['laser_power_W', 'welding_speed_mm_s']
targets = ['weld_bead_width_mm', 'interface_width_mm', 'penetration_depth_mm']
required = features + targets
missing = [c for c in required if c not in df_raw.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}. Use the standardized Notebook 03 CSV.')

df = df_raw.copy()
for col in required:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=required).copy()
df['condition_group'] = (df['laser_power_W'].astype(str) + 'W_' + df['welding_speed_mm_s'].astype(str) + 'mm_s')
all_zero = df[targets].eq(0).all(axis=1)
print('Complete source rows:', len(df))
print('All-three-output-zero rows retained in source input but excluded from default regression:', int(all_zero.sum()))
df = df.loc[~all_zero].copy()
print('Eligible rows:', len(df), '| independent process conditions:', df['condition_group'].nunique())
display(df.groupby('condition_group').size().rename('replicate_records').reset_index())
if df['condition_group'].nunique() < 3:
    raise ValueError('Too few independent process conditions for grouped evaluation.')

## 2. Leave-one-condition-out evaluation

Each fold holds out every replicate from one power–speed condition. Compare a training-mean baseline, standardized Ridge regression and Extra Trees. No hyperparameter tuning is performed.

In [ ]:
cv = LeaveOneGroupOut()
groups = df['condition_group']
models = {
    'Mean baseline': DummyRegressor(strategy='mean'),
    'Ridge regression': make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'Extra Trees': ExtraTreesRegressor(n_estimators=300, min_samples_leaf=2, random_state=42)
}
fold_rows, oof_rows = [], []
for target in targets:
    X, y = df[features], df[target].astype(float)
    for model_name, model in models.items():
        for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
            fitted = model
            fitted.fit(X.iloc[train_idx], y.iloc[train_idx])
            pred = fitted.predict(X.iloc[test_idx])
            fold_rows.append({
                'target': target, 'model': model_name, 'fold': fold,
                'held_out_condition': groups.iloc[test_idx].iloc[0],
                'train_records': len(train_idx), 'test_records': len(test_idx),
                'MAE_mm': mean_absolute_error(y.iloc[test_idx], pred),
                'RMSE_mm': np.sqrt(mean_squared_error(y.iloc[test_idx], pred)),
                'R2_fold': r2_score(y.iloc[test_idx], pred) if len(test_idx) > 1 else np.nan
            })
            for idx, actual, prediction in zip(test_idx, y.iloc[test_idx], pred):
                oof_rows.append({
                    'target': target, 'model': model_name, 'row_index': int(idx),
                    'record_id': str(df.iloc[idx]['record_id']) if 'record_id' in df.columns else str(df.index[idx]),
                    'condition_group': groups.iloc[idx], 'actual_mm': float(actual),
                    'predicted_mm': float(prediction), 'residual_mm': float(actual-prediction)
                })
fold_df, oof_df = pd.DataFrame(fold_rows), pd.DataFrame(oof_rows)
print('FOLD METRICS')
display(fold_df.round(4))
pooled_rows = []
for (target, model_name), part in oof_df.groupby(['target','model']):
    pooled_rows.append({'target':target,'model':model_name,
        'MAE_mm':mean_absolute_error(part.actual_mm,part.predicted_mm),
        'RMSE_mm':np.sqrt(mean_squared_error(part.actual_mm,part.predicted_mm)),
        'R2_pooled':r2_score(part.actual_mm,part.predicted_mm)})
pooled_df = pd.DataFrame(pooled_rows)
print('POOLED OUT-OF-FOLD METRICS')
display(pooled_df.round(4))
print('FOLD METRIC MEAN AND STANDARD DEVIATION')
display(fold_df.groupby(['target','model'])[['MAE_mm','RMSE_mm','R2_fold']].agg(['mean','std']).round(4))

## 3. Diagnostic plots and exports

Pooled out-of-fold predictions combine predictions from models that did not train on the corresponding process condition. Interpret metrics cautiously: there are only seven held-out conditions.

In [ ]:
for target in targets:
    part = oof_df[(oof_df.target == target) & (oof_df.model == 'Extra Trees')]
    plt.figure(figsize=(5.5,5))
    plt.scatter(part.actual_mm, part.predicted_mm)
    lo = min(part.actual_mm.min(), part.predicted_mm.min())
    hi = max(part.actual_mm.max(), part.predicted_mm.max())
    plt.plot([lo,hi],[lo,hi],'--')
    plt.xlabel(f'Measured {target} (mm)')
    plt.ylabel('Out-of-fold predicted value (mm)')
    plt.title(f'Extra Trees — {target}')
    plt.tight_layout()
    plt.show()

out_dir = Path('notebook09_conventional_ml_outputs')
out_dir.mkdir(exist_ok=True)
fold_df.to_csv(out_dir/'conventional_fold_metrics.csv', index=False)
pooled_df.to_csv(out_dir/'conventional_pooled_oof_metrics.csv', index=False)
oof_df.to_csv(out_dir/'conventional_oof_predictions.csv', index=False)
df.to_csv(out_dir/'conventional_eligible_model_data.csv', index=False)
import zipfile
zip_path = Path('notebook09_conventional_ml_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for file in out_dir.glob('*.csv'):
        z.write(file, arcname=file.name)
print('Created:', zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('ZIP saved in the current working directory.')

## Interpretation rules

- Use the pooled out-of-fold MAE and RMSE as the primary descriptive comparison; report fold variability and the seven independent condition groups.
- Fold-level R² is not useful when a held-out group contains only three replicates and may be unstable. Do not interpret it in isolation.
- The six all-zero geometry rows are not discarded from the master dataset; they are excluded here only from the default geometry-regression experiment pending source interpretation.
- This is within-source, process-condition-grouped validation, not external validation across studies, laboratories or equipment.
- Do not claim causal effects, an optimum welding window or production-ready prediction. The dataset has only 21 eligible rows and two process inputs.

The output CSVs preserve fold metrics, pooled predictions and the exact eligible modelling table for review.